In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import joblib
import time
from pathlib import Path


# sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_score, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin, clone

# Importações locais
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.preprocess_utils_tic import preprocessador_titanic
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))


#Processo finalizado em: 11:07:05


## 2. Dataload & Preprocessamento com joblib

In [4]:
BASE = Path.cwd().parent   
# =====================================================
# ⚙️ 0. carregamento dos preprocessador 
# =====================================================
temp = joblib.load(BASE /'src'/'preprocess_Titanic_v1.2.joblib')
PP2=temp['preprocessador']

# # =====================================================
# # 📁 1. Leitura dos dados & Separação das bases
# # =====================================================

DATA_DIR = BASE / "data" / "raw"
# X_train = pd.read_csv(DATA_DIR / "X_train_raw.csv").reset_index(drop=True)
# X_val  = pd.read_csv(DATA_DIR / "X_test_raw.csv")
# y_train = pd.read_csv(DATA_DIR / "y_train_raw.csv").values.ravel()
# y_val  = pd.read_csv(DATA_DIR / "y_test_raw.csv")

#full
TARGET = 'Survived'
dfo = pd.read_csv(DATA_DIR / "train.csv").drop(columns='PassengerId')
X_train = dfo.drop(columns=TARGET)
y_train = dfo[TARGET]


# =====================================================
# 📁 2. Modelo e pipeline
# =====================================================
DATA_MODEL = BASE / "models"
pipe_RF3 = joblib.load(DATA_MODEL /'modelo_RF_final_randsearch.roc_auc_v12.joblib')
pipe_RF3.named_steps # informações sobre a pipeline

{'preprocess': preprocessador_titanic(),
 'model': RandomForestClassifier(max_depth=30, max_features=0.7, max_leaf_nodes=100,
                        max_samples=0.7, min_samples_split=4,
                        min_weight_fraction_leaf=0.05979712296915085,
                        n_estimators=511, n_jobs=-1, random_state=42)}

In [5]:
# =====================================================
# 3.Treino & Teste
# =====================================================
#cv_scores = cross_val_score(pipe_RF3 , X_train, y_train, cv=10)
pipe_RF3.fit(X_train, y_train)
# y_pred=pipe_RF3.predict(X_val) #validação


# print(f"{'='*70}")
# print(f"🎯 Random Forest 2 | cvscores : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
# print(f"{'='*70}")
# print(f"📊 **Acurácia no Teste**: {accuracy_score(y_val, y_pred):.4f}")
# print(f"\n📋 **Relatório de Classificação**:")
# print(classification_report(y_val, y_pred))
# cm=confusion_matrix(y_val, y_pred)
# print(f"🎯 **Matriz de Confusão**:")
# print(f"               Previsto 0   Previsto 1")
# print(f"Real 0         {cm[0,0]:<11} {cm[0,1]:<11}")
# print(f"Real 1         {cm[1,0]:<11} {cm[1,1]:<11}")
# print(f"{'─'*70}")

Pipeline(steps=[('preprocess', preprocessador_titanic()),
                ('model',
                 RandomForestClassifier(max_depth=30, max_features=0.7,
                                        max_leaf_nodes=100, max_samples=0.7,
                                        min_samples_split=4,
                                        min_weight_fraction_leaf=0.05979712296915085,
                                        n_estimators=511, n_jobs=-1,
                                        random_state=42))])

## 3. Kaggle

In [6]:
# =====================================================
# Submissão Kaggle
# =====================================================
base = pd.read_csv("/home/akel/PycharmProjects/Kaggle/Titanic/data/raw/test.csv")

id_test = base["PassengerId"]

x_test=base.drop(columns='PassengerId')

# Previsão
y_pred=pipe_RF3.predict(x_test)

#y_probs = pipe_RF3.predict_proba(x_test)[:, 1]
#y_pred = (y_probs >= best_threshold).astype(int)

#  DataFrame de submissão
submission = pd.DataFrame({
    "PassengerId": id_test,
    "Survived": y_pred
})
submission_path = ("/home/akel/PycharmProjects/Kaggle/Titanic/data/processed/submission_RF_final_randsearch.roc_auc_v12F.csv")
submission.to_csv(submission_path, index=False)
print("✅ Arquivo de submissão salvo com sucesso!")
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))

✅ Arquivo de submissão salvo com sucesso!

#Processo finalizado em: 11:10:00
